# APTOS 2019 — ResNet-50 ImageNet + full fine-tune

ImageNet weights, all layers trainable. The pretrained backbone uses a small learning rate, the new head a larger one. Reads the output folder of the preprocessing notebook (`aptos_processed`). The best epoch is chosen by validation QWK; the test set is only used at the end.

In [ ]:
import copy, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.utils.class_weight import compute_class_weight

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms as T

MODE = "finetune"
SEED = 42
NUM_CLASSES = 5
CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative DR"]

TRAIN_SET = "train"            # "train" = balanced (oversampled PNGs)  |  "train_original" = original + class weights
BATCH_SIZE = 8                 # images stay at 1024x1024 (no resize): 4 per GPU on 2x T4
NUM_WORKERS = 4
EPOCHS = 20
LR_BACKBONE, LR_HEAD = 1e-4, 1e-3

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPUs:", torch.cuda.device_count())

## Load the preprocessed dataset

In [ ]:
def find_data_root():
    candidates = [Path("/kaggle/working/aptos_processed")]
    candidates += [p.parent.parent for p in Path("/kaggle/input").rglob("train_original.csv")]
    for root in candidates:
        if (root / "metadata" / "train_original.csv").is_file():
            return root
    raise FileNotFoundError("Run the preprocessing notebook first, "
                            "or add its output as an input dataset.")

DATA_ROOT = find_data_root()
print("Dataset:", DATA_ROOT)

def load_meta(csv_name, folder):
    frame = pd.read_csv(DATA_ROOT / "metadata" / f"{csv_name}.csv")
    # rebuild the paths under the current root (the CSV keeps the paths of the run that created it)
    frame["path"] = [str(DATA_ROOT / folder / str(c) / Path(f).name)
                     for c, f in zip(frame.diagnosis, frame.filepath)]
    assert all(Path(p).exists() for p in frame["path"]), f"Missing images for {csv_name}"
    return frame

train_meta = load_meta("train_original" if TRAIN_SET == "train_original" else "train_balanced", TRAIN_SET)
val_meta = load_meta("val", "val")
test_meta = load_meta("test", "test")

display(pd.DataFrame({"train": train_meta.diagnosis.value_counts().sort_index(),
                      "val": val_meta.diagnosis.value_counts().sort_index(),
                      "test": test_meta.diagnosis.value_counts().sort_index()}))

In [ ]:
# Class weights replace oversampling when training on the original images.
if TRAIN_SET == "train_original":
    w = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_CLASSES),
                             y=train_meta.diagnosis.to_numpy())
    class_weights = torch.tensor(w, dtype=torch.float32).to(DEVICE)
    print("Class weights:", np.round(w, 2))
else:
    class_weights = None

norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])   # ImageNet stats
train_tf = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20, fill=(128, 128, 128)),   # gray = background of the preprocessed images
    T.ToTensor(), norm,
])
eval_tf = T.Compose([T.ToTensor(), norm])

def gray_outside_circle(img):
    # The augmented PNGs were saved with a black background, the originals have a gray (128) one.
    # Paint the outside of the retina circle gray so that both look the same.
    arr = np.array(img)
    h, w = arr.shape[:2]
    yy, xx = np.ogrid[:h, :w]
    arr[(yy - h // 2) ** 2 + (xx - w // 2) ** 2 > int(0.5 * min(h, w)) ** 2] = 128
    return Image.fromarray(arr)

class RetinaDataset(Dataset):
    def __init__(self, meta, tf):
        self.paths = meta["path"].tolist()
        self.labels = meta["diagnosis"].astype(int).tolist()
        self.augmented = meta["is_augmented"].astype(bool).tolist()
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        if self.augmented[i]:
            img = gray_outside_circle(img)
        return self.tf(img), self.labels[i]

def make_loader(meta, tf, shuffle):
    return DataLoader(RetinaDataset(meta, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=shuffle)

train_loader = make_loader(train_meta, train_tf, True)
val_loader = make_loader(val_meta, eval_tf, False)
test_loader = make_loader(test_meta, eval_tf, False)

## Model and training loop

In [ ]:
def build_resnet50(pretrained):
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)   # new 5-class head
    model = model.to(DEVICE)
    if torch.cuda.device_count() > 1:                         # e.g. Kaggle T4 x2
        model = nn.DataParallel(model)
    return model

use_amp = DEVICE == "cuda"
criterion = nn.CrossEntropyLoss(weight=class_weights)   # used for training
eval_criterion = nn.CrossEntropyLoss()                  # unweighted, for validation loss

def metrics(preds, labels, loss):
    return {"loss": loss,
            "acc": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average="macro", labels=range(NUM_CLASSES), zero_division=0),
            "qwk": cohen_kappa_score(labels, preds, weights="quadratic")}

@torch.no_grad()
def evaluate(model, data_loader):
    model.eval()
    preds, labels, total_loss = [], [], 0.0
    for x, y in tqdm(data_loader, desc="evaluate", leave=False):
        with torch.autocast(device_type="cuda", enabled=use_amp):
            out = model(x.to(DEVICE, non_blocking=True))
        total_loss += eval_criterion(out.float(), y.to(DEVICE)).item() * len(y)
        preds += out.argmax(1).cpu().tolist()
        labels += y.tolist()
    return preds, labels, total_loss / len(labels)

def train(model, optimizer, epochs, name):
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    history, best_qwk, best_state, best_epoch = [], -1, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        preds, labels, total_loss = [], [], 0.0
        bar = tqdm(train_loader, desc=f"[{name}] epoch {epoch}/{epochs}", leave=False)
        for x, y in bar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                out = model(x)
                loss = criterion(out.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            bar.set_postfix(loss=f"{loss.item():.3f}")
            total_loss += loss.item() * len(y)
            preds += out.argmax(1).cpu().tolist()
            labels += y.cpu().tolist()
        scheduler.step()

        tr = metrics(preds, labels, total_loss / len(labels))
        va = metrics(*evaluate(model, val_loader))
        history.append({"train": tr, "val": va})
        print(f"[{name}] epoch {epoch}/{epochs}  "
              f"train loss {tr['loss']:.3f} acc {tr['acc']:.3f} | "
              f"val loss {va['loss']:.3f} acc {va['acc']:.3f} F1 {va['f1']:.3f} QWK {va['qwk']:.3f}")

        if va["qwk"] > best_qwk:
            best_qwk, best_epoch = va["qwk"], epoch
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    print(f"[{name}] best epoch {best_epoch} (val QWK {best_qwk:.3f})")
    return history

## Train — ImageNet weights, all layers trainable

In [ ]:
model = build_resnet50(pretrained=True)
base = model.module if isinstance(model, nn.DataParallel) else model
backbone = [p for n, p in base.named_parameters() if not n.startswith("fc.")]
optimizer = torch.optim.AdamW([{"params": backbone, "lr": LR_BACKBONE},
                               {"params": base.fc.parameters(), "lr": LR_HEAD}], weight_decay=1e-4)
history = train(model, optimizer, EPOCHS, name=MODE)
torch.save(model.state_dict(), f"/kaggle/working/resnet50_{MODE}.pt")

## Training curves (train vs val)

In [ ]:
epochs = range(1, len(history) + 1)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, key, label in zip(axes, ["loss", "acc", "f1", "qwk"], ["Loss", "Accuracy", "Macro-F1", "QWK"]):
    ax.plot(epochs, [h["train"][key] for h in history], label="train")
    ax.plot(epochs, [h["val"][key] for h in history], label="val")
    ax.set(title=label, xlabel="epoch")
    ax.legend()
plt.tight_layout()
plt.show()

## Test results

In [ ]:
preds, labels, loss = evaluate(model, test_loader)
result = metrics(preds, labels, loss)

print("true     :", np.bincount(labels, minlength=NUM_CLASSES))
print("predicted:", np.bincount(preds, minlength=NUM_CLASSES))
print(classification_report(labels, preds, labels=range(NUM_CLASSES),
                            target_names=CLASS_NAMES, digits=3, zero_division=0))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(labels, preds, labels=range(NUM_CLASSES)),
                       display_labels=CLASS_NAMES
                       ).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
ax.set_title(f"{MODE} — test confusion matrix")
plt.tight_layout()
plt.show()

# keep the numbers so the two models can be compared later
with open(f"/kaggle/working/results_{MODE}.json", "w") as f:
    json.dump({"test": result, "history": history}, f, indent=2)

pd.Series(result).round(3)